# Color-invariant saree retrieval

This notebook runs the repo. It does not copy the training code. Upload the repository, or run it from the project root.

The task is identification and verification of a saree motif. A gallery image of the same design in another palette should rank high. A different motif should rank low even when the palette matches. That is a similarity problem, so the model is an embedding trained with supervised contrastive loss, not a softmax over a fixed list of saree classes.

There are no real colorway labels, so a second palette is a Lab chroma rotation of the same photo. Photos are held out entirely. Within a held-out photo the gallery hue and the query hue are disjoint. Training on two real colorways and testing on a third is not possible with this corpus.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("working directory:", Path.cwd())

APPROACH_NOTE = (
    "MobileNetV3-Small maps a 224px crop to a 128-d L2-normalized embedding. "
    "Each photo is one design. Positives are its crops under Lab hue rotation and saturation change; "
    "other designs in the batch are negatives. Supervised contrastive loss, PK sampling, flip and mild blur. "
    "Cosine similarity ranks the gallery and verifies pairs. ImageNet initialization, disclosed. "
    "Real colorway labels are absent, so palettes are synthetic and the eval says so."
)
print(f"approach note: {len(APPROACH_NOTE)} characters")
assert len(APPROACH_NOTE) <= 500
assert "synthetic" in APPROACH_NOTE


## Data

The Drive corpus has no design id in the filename. Each source photo is one design. `normal_sarees` was empty, and the Kaggle weave-class set is not used. Other palettes are produced in Lab so lightness, and therefore the motif edges, stay put. One view in each training batch shares a hue across designs, so a different motif in the same palette is a negative.


In [ ]:
from src.data import list_images, split_designs

root = Path("data/sarees")
if not any(root.rglob("*.jpg")):
    from src.download import download_corpus

    download_corpus(root)
paths = list_images(root)
split = split_designs(paths, seed=42)
print(f"photos {len(paths)}  train {len(split.train)}  val {len(split.val)}  test {len(split.test)}")
print("split: 60/20/20 by photo, seed 42. Crops of one photo stay in one split.")
print("gallery: center 224, Lab hue +40. query: shifted 48px, mirrored, Lab hue +160.")


## Training

The color run is the submitted model. The no-color run is the ablation that never changes palette. If a checkpoint is already on disk, this cell loads it. Otherwise it trains.

```bash
python -m src.train --data data/sarees --out runs/color
python -m src.train --data data/sarees --out runs/nocolor --no-color-aug
```


In [ ]:
import torch


def ensure_checkpoint(out: str, extra: list[str]) -> None:
    ckpt = Path(out) / "best.pt"
    if not ckpt.exists():
        cmd = [sys.executable, "-m", "src.train", "--data", "data/sarees", "--out", out, *extra]
        print("training:", " ".join(cmd))
        subprocess.check_call(cmd)
    saved = torch.load(ckpt, map_location="cpu", weights_only=False)
    print(f"{out}: epoch {saved.get('epoch')}  embed_dim {saved['args'].get('embed_dim')}")


ensure_checkpoint("runs/color", [])
ensure_checkpoint("runs/nocolor", ["--no-color-aug"])


## Evaluation

Identification reports Rank-1, Rank-5, and mAP on the held-out photos. Verification reports ROC-AUC, equal-error rate, and the true-accept rate when the threshold is the one chosen on validation. The same-palette stress test recolors every gallery photo to the query hue. Rank-1 there means the motif matched, not the color.

The ImageNet row is the same MobileNetV3-Small backbone with no fine-tuning. The embedding is the L2-normalized pooled feature. The untrained projection is not used.


In [ ]:
import json

commands = [
    [sys.executable, "-m", "src.eval", "--checkpoint", "runs/color/best.pt"],
    [sys.executable, "-m", "src.eval", "--checkpoint", "runs/nocolor/best.pt"],
    [sys.executable, "-m", "src.eval", "--imagenet"],
]
for cmd in commands:
    subprocess.check_call(cmd)


def report_line(name: str, path: str) -> None:
    payload = json.loads(Path(path).read_text())
    test = payload["test"]
    ident = test["cross_palette_identification"]
    ver = test["cross_palette_verification"]
    stress = test["same_palette_stress"]
    threshold = ver.get("val_threshold", ver["eer_threshold"])
    print(
        f"{name:16}  R@1 {ident['rank1']:.3f}  R@5 {ident['rank5']:.3f}  "
        f"mAP {ident['mAP']:.3f}  AUC {ver['roc_auc']:.3f}  EER {ver['eer']:.3f}  "
        f"stress R@1 {stress['rank1']:.3f}  val threshold {threshold:.3f}"
    )
    if "tar_at_val_threshold" in ver:
        print(
            f"{'':16}  TAR {ver['tar_at_val_threshold']:.3f}  "
            f"FAR {ver['far_at_val_threshold']:.3f}  at the val threshold"
        )


print("held-out test, synthetic palettes")
report_line("color SupCon", "runs/color/eval.json")
report_line("no color aug", "runs/nocolor/eval.json")
report_line("ImageNet frozen", "runs/imagenet/eval.json")


## Retrieval strips

Each strip starts with the query crop at hue +160°, shifted and mirrored. The next five images are the top gallery crops at hue +40°. The score is cosine similarity. The true design is the gallery image from the same photo.


In [ ]:
import torch
from IPython.display import display
from PIL import Image

from src.data import GALLERY_HUE, QUERY_HUE
from src.eval import device_of, embed_images, load_model, region_views

device = device_of()
model, _saved = load_model(Path("runs/color/best.pt"), device)
gallery_images = region_views(split.test, GALLERY_HUE, "gallery")
query_images = region_views(split.test, QUERY_HUE, "query")
gallery = embed_images(model, gallery_images, device)
query = embed_images(model, query_images, device)
similarity = query @ gallery.T


def contact(images: list[Image.Image]) -> Image.Image:
    gap = 8
    width, height = images[0].size
    canvas = Image.new("RGB", (width * len(images) + gap * (len(images) - 1), height), (255, 255, 255))
    x = 0
    for image in images:
        canvas.paste(image, (x, 0))
        x += width + gap
    return canvas


for index in range(min(3, len(split.test))):
    order = torch.argsort(similarity[index], descending=True)[:5].tolist()
    scores = [float(similarity[index, j]) for j in order]
    print(f"query {split.test[index].name}  correct={order[0] == index}")
    print("  " + "  ".join(f"{split.test[j].name} {score:.3f}" for j, score in zip(order, scores)))
    display(contact([query_images[index], *[gallery_images[j] for j in order]]))


## Efficiency

Parameter count, multiply-adds in convolutions and linear layers, embedding size, and single-image latency for the submitted encoder.


In [ ]:
subprocess.check_call([sys.executable, "-m", "src.efficiency", "--checkpoint", "runs/color/best.pt"])
print(Path("runs/color/efficiency.json").read_text())
